In [1]:
!pip install -U transformers shap

from google.colab import drive
drive.mount('/content/drive')

LOAD_DIR = "/content/drive/MyDrive/bert_reviews_model"  # твоя папка с моделью

from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import pandas as pd
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(LOAD_DIR)
model = AutoModelForSequenceClassification.from_pretrained(LOAD_DIR).to(device)
model.eval()

print("Модель и токенайзер загружены.")

Mounted at /content/drive
Модель и токенайзер загружены.


In [2]:
sushi_path = "/content/sushi_reviews.csv"
df_sushi = pd.read_csv(sushi_path)

print(df_sushi.head())
print(df_sushi.columns)

   id        category                                               text
0   1  3_stars_manual  Здравствуйте, уважаемые читатели! Иногда по пя...
1   2  2_stars_manual  Привет любителям японской кухни и нашего повсе...
2   3  1_stars_manual  Здравствуйте! Ну что ж, вот и я решила попробо...
3   4  3_stars_manual  Всем привет! Про доставку суши и роллов со так...
4   5  1_stars_manual  Решили как-то с семьей разнообразить вкусовые ...
Index(['id', 'category', 'text'], dtype='object')


In [3]:
def predict_batch(texts, max_length=256):
    if isinstance(texts, str):
        texts = [texts]
    texts = [str(t) for t in texts]

    enc = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        outputs = model(**enc)
        probs = torch.softmax(outputs.logits, dim=1).cpu().numpy()
        preds = np.argmax(probs, axis=1)
    return preds, probs

In [4]:
preds, probs = predict_batch(df_sushi["text"].tolist())

df_sushi["pred_label"] = preds              # 0 = плохой, 1 = хороший
df_sushi["prob_neg"] = probs[:, 0]
df_sushi["prob_pos"] = probs[:, 1]

df_sushi.head()

,id,category,text,pred_label,prob_neg,prob_pos
0,1,3_stars_manual,"Здравствуйте, уважаемые читатели! Иногда по пя...",0,0.999843,0.000157
1,2,2_stars_manual,Привет любителям японской кухни и нашего повсе...,0,0.999657,0.000343
2,3,1_stars_manual,"Здравствуйте! Ну что ж, вот и я решила попробо...",0,0.999853,0.000147
3,4,3_stars_manual,Всем привет! Про доставку суши и роллов со так...,0,0.999854,0.000146
4,5,1_stars_manual,Решили как-то с семьей разнообразить вкусовые ...,0,0.999855,0.000145


In [5]:
import shap

def predict_pos(texts, max_length=256):
    """
    Возвращает P(class=1) — вероятность положительного отзыва.
    Удобно для SHAP: одна целевая переменная.
    """
    if isinstance(texts, str):
        texts = [texts]

    processed = []
    for t in texts:
        if isinstance(t, list):      # SHAP может передать список токенов
            processed.append(" ".join(t))
        else:
            processed.append(str(t))

    enc = tokenizer(
        processed,
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        outputs = model(**enc)
        probs = torch.softmax(outputs.logits, dim=1).cpu().numpy()
        pos_probs = probs[:, 1]
    return pos_probs

In [6]:
masker = shap.maskers.Text(tokenizer=r"\W+")
explainer = shap.Explainer(predict_pos, masker=masker)

In [7]:
from collections import defaultdict

RUS_STOPWORDS = {
    "и", "в", "во", "на", "но", "что", "как", "к", "ко", "от", "до", "за",
    "из", "у", "по", "а", "не", "это", "так", "же", "то", "с", "со"
}

def pros_cons_from_reviews(texts, n_samples=50, top_k=20, min_token_len=3):
    """
    texts      — список текстов отзывов ОДНОГО товара
    n_samples  — максимум отзывов для SHAP (ограничиваем, чтобы не ждать час)
    top_k      — сколько топ-слов вернуть
    """
    if len(texts) == 0:
        raise ValueError("Нет отзывов для анализа.")

    # подвыборка для скорости
    import random
    if len(texts) > n_samples:
        texts = random.sample(list(texts), n_samples)

    # считаем shap-значения
    shap_values = explainer(list(texts))

    pos_scores = defaultdict(float)  # слова, повышающие P(positive)
    neg_scores = defaultdict(float)  # слова, снижающие P(positive)

    for ex in shap_values:           # ex: shap.Explanation для одного отзыва
        tokens = ex.data             # токены (слова/символы)
        values = ex.values           # shap-вклады для каждого токена

        for tok, v in zip(tokens, values):
            tok = str(tok).lower().strip()
            if len(tok) < min_token_len:
                continue
            if tok in RUS_STOPWORDS:
                continue

            if v > 0:
                pos_scores[tok] += float(v)
            elif v < 0:
                neg_scores[tok] += float(-v)  # модуль

    pros = sorted(pos_scores.items(), key=lambda x: x[1], reverse=True)[:top_k]
    cons = sorted(neg_scores.items(), key=lambda x: x[1], reverse=True)[:top_k]

    return pros, cons

In [ ]:
texts_sushi = df_sushi["text"].astype(str).tolist()
pros, cons = pros_cons_from_reviews(texts_sushi, n_samples=80, top_k=20)

  0%|          | 0/498 [00:00<?, ?it/s]

PartitionExplainer explainer:   1%|▏         | 1/80 [00:00<?, ?it/s]

  0%|          | 0/498 [00:00<?, ?it/s]

PartitionExplainer explainer:   4%|▍         | 3/80 [04:57<2:11:23, 102.38s/it]

  0%|          | 0/498 [00:00<?, ?it/s]

...

PartitionExplainer explainer:  32%|███▎      | 26/80 [1:09:39<2:44:40, 182.97s/it]

  0%|          | 0/498 [00:00<?, ?it/s]

PartitionExplainer explainer:  34%|███▍      | 27/80 [1:17:01<3:50:17, 260.72s/it]

  0%|          | 0/498 [00:00<?, ?it/s]

PartitionExplainer explainer:  35%|███▌      | 28/80 [1:18:00<2:53:27, 200.14s/it]

In [ ]:
def print_pros_cons(pros, cons):
    print("⭐ Что клиенты любят (плюсы):")
    for word, score in pros:
        print(f"- {word}")

    print("\n⚡ Что клиентов раздражает (минусы):")
    for word, score in cons:
        print(f"- {word}")

In [ ]:
print_pros_cons(pros, cons)

In [ ]:
def build_llm_prompt(product_name, category, pros, cons, tone="дружелюбный, но профессиональный"):
    pros_words = [w for w, _ in pros]
    cons_words = [w for w, _ in cons]

    prompt = f"""
Ты — опытный маркетолог и копирайтер. Товар: "{product_name}", категория: {category}.

На основе анализа реальных отзывов клиентов известны следующие особенности.

Сильные стороны (то, что клиенты чаще всего хвалят):
- {", ".join(pros_words) if pros_words else "нет данных"}

Слабые стороны и потенциальные риски (то, чем люди часто недовольны):
- {", ".join(cons_words) if cons_words else "нет данных"}

Задача:
1. Придумай 3 варианта короткого рекламного текста для соцсетей (2–3 предложения),
   стиль — {tone}.
2. В каждом тексте:
   - сделай акцент на сильных сторонах товара;
   - мягко закрой типичные возражения клиента, НЕ обещая того, что мы не контролируем
     (например, если есть проблемы с доставкой, не обещай "молниеносная доставка").
3. Добавь по одному короткому слогану (до 6–7 слов) к каждому варианту.

Ответ выдай в структурированном виде:
Вариант 1:
Текст: ...
Слоган: ...

Вариант 2:
...
"""
    return prompt